# Vase Feature Extraction And Collinearity Workflow
This notebook runs end-to-end extraction, then removes redundant features using correlation-based clustering.

## How To Use
1. Run cells from top to bottom in order.
2. If extraction is already complete, you may skip the extraction cell and begin from the analysis section.
3. Re-run analysis cells after changing thresholds (for example `corr_threshold`).

## What This Notebook Produces
- A compiled feature matrix from raw vase geometry and metadata.
- A reduced (purged) feature set with lower collinearity.
- Diagnostic visualisations to support interpretation and team review.

In [1]:
import importlib
import tqdm as notebook_tqdm
import pandas as pd
import os
from scipy.stats import spearmanr
from scipy.spatial.distance import squareform
import scipy.cluster.hierarchy as sch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from pathlib import Path
from function_modules.main_extractor import execute_master_pipeline_from_dataframe

In [2]:
TARGET_SCURVES_DIR = "hubner_lines_0326"
OUTPUT_DIR = "outdir/gpt-5.4-270526"
OUTPUT_MATRIX_PATH = f"{OUTPUT_DIR}/master_feature_matrix_compiled.csv"
OUTPUT_MATRIX_PARQUET_PATH = f"{OUTPUT_DIR}/master_feature_matrix_compiled.parquet"
CHECKPOINT_INTERVAL_SECONDS = 30
BATCH_SIZE = 100
USE_TQDM = True

# Keep batches small — with 250-point XY profiles each participant is ~14k rows.
# 25 participants ≈ 340k rows per batch; safe on most laptops.
PARTICIPANTS_PER_BATCH = 25

os.makedirs(OUTPUT_DIR, exist_ok=True)

## Step 1: Run Extraction From Master Parquet

The pipeline expects these columns (mapped automatically if needed):

| Parquet column | Pipeline name |
|---|---|
| `gen` or `generation_key` | `generation` |
| `session_id` | `user_id` |
| `is_selected_vase` | `gen_selected_vase` |
| `vase`, `x`, `y` | (unchanged) |

Any extra columns (`rep`, `rt_tokens`, `model`, etc.) are passed through to the output matrix.

In [3]:
XY_PARQUET_PATH       = "../ai_beauty/pca_llm_results/processed/ai_xy_data.parquet"
METADATA_PARQUET_PATH = "../ai_beauty/pca_llm_results/processed/ai_xy_data.parquet"
PARTICIPANTS_PER_BATCH = 500  # tune down if RAM is tight, up to go faster

# Read only the participant_id column to plan batches — no x/y data loaded yet
_pid_col = pd.read_parquet(XY_PARQUET_PATH, columns=['participant_id'])
_all_participants = _pid_col['participant_id'].unique()
del _pid_col

_n_batches = (len(_all_participants) + PARTICIPANTS_PER_BATCH - 1) // PARTICIPANTS_PER_BATCH
print(f"Total unique participants : {len(_all_participants):,}")
print(f"Will process in {_n_batches} batches of up to {PARTICIPANTS_PER_BATCH} participants")

Total unique participants : 606
Will process in 2 batches of up to 500 participants


In [4]:
# ── Starting-point diagnostic ─────────────────────────────────────────────────
# Checks geometric preconditions on a 50-participant sample from the parquet
# rather than loading the full dataset, keeping memory under control.

_diag_sample_participants = _all_participants[:50].tolist()
_xy = pd.read_parquet(
    XY_PARQUET_PATH,
    columns=['participant_id', 'generation_key', 'vase', 'x', 'y'],
    filters=[('participant_id', 'in', _diag_sample_participants)],
)

# 1. Point count per vase
EXPECTED_PTS = 250
_pts = _xy.groupby(['participant_id', 'generation_key', 'vase']).size()
_wrong_pts = _pts[_pts != EXPECTED_PTS]

# 2. NaN / inf in coordinates
_bad_x = _xy['x'].isna() | np.isinf(_xy['x'])
_bad_y = _xy['y'].isna() | np.isinf(_xy['y'])
_n_bad_coords = int((_bad_x | _bad_y).sum())

# 3. x at y_min — base of each half-profile should sit on centreline (x ≈ 0)
X_BASE_TOLERANCE = 0.15
_base_rows = _xy.loc[_xy.groupby(['participant_id', 'generation_key', 'vase'])['y'].idxmin()]
_x_base = _base_rows['x'].abs()
_offcentre = _x_base[_x_base > X_BASE_TOLERANCE]

# 4. Profile height — catch degenerate flat profiles
MIN_HEIGHT = 0.5
_vase_y = _xy.groupby(['participant_id', 'generation_key', 'vase'])['y'].agg(['min', 'max'])
_vase_y['height'] = _vase_y['max'] - _vase_y['min']
_flat = _vase_y[_vase_y['height'] < MIN_HEIGHT]

# ── Report ───────────────────────────────────────────────────────────────────
print(f"Diagnostic sample        : {len(_diag_sample_participants)} participants")
print(f"Vases in sample          : {len(_pts):,}")
print()

checks = [
    ("Point count (expected 250)",  len(_wrong_pts) == 0,  f"{len(_wrong_pts):,} vases with wrong point count"),
    ("No NaN / inf in coordinates", _n_bad_coords == 0,    f"{_n_bad_coords:,} rows with bad x or y values"),
    (f"Base on centreline (|x| < {X_BASE_TOLERANCE})", len(_offcentre) == 0, f"{len(_offcentre):,} vases off-centre"),
    (f"No flat profiles (height ≥ {MIN_HEIGHT})", len(_flat) == 0, f"{len(_flat):,} degenerate profiles"),
]

all_passed = True
for label, passed, detail in checks:
    status = "PASS" if passed else "WARN"
    if not passed:
        all_passed = False
    print(f"  [{status}] {label}")
    if not passed:
        print(f"         → {detail}")

print()
print(f"  y range  : [{_vase_y['min'].min():.3f}, {_vase_y['max'].max():.3f}]")
print(f"  height   : mean={_vase_y['height'].mean():.3f}  std={_vase_y['height'].std():.3f}")
print(f"  x_base   : mean={_x_base.mean():.4f}  max={_x_base.max():.4f}")
print()
print("Sample checks passed — safe to proceed." if all_passed else "WARNs above — review before running.")
del _xy, _pts, _vase_y, _base_rows

Diagnostic sample        : 50 participants
Vases in sample          : 2,750

  [PASS] Point count (expected 250)
  [PASS] No NaN / inf in coordinates
  [WARN] Base on centreline (|x| < 0.15)
         → 1,774 vases off-centre
  [WARN] No flat profiles (height ≥ 0.5)
         → 1 degenerate profiles

  y range  : [-3.061, 3.067]
  height   : mean=3.046  std=0.936
  x_base   : mean=0.5969  max=3.0626

WARNs above — review before running.


In [5]:
import gc

_n_batches = (len(_all_participants) + PARTICIPANTS_PER_BATCH - 1) // PARTICIPANTS_PER_BATCH

for _batch_num, _batch_start in enumerate(range(0, len(_all_participants), PARTICIPANTS_PER_BATCH), 1):
    _batch_participants = _all_participants[_batch_start : _batch_start + PARTICIPANTS_PER_BATCH].tolist()
    print(f"\n{'='*60}")
    print(f"Participant batch {_batch_num}/{_n_batches}: {len(_batch_participants)} participants")
    print(f"{'='*60}")

    _batch_df = pd.read_parquet(
        XY_PARQUET_PATH,
        filters=[('participant_id', 'in', _batch_participants)],
    )

    execute_master_pipeline_from_dataframe(
        df=_batch_df,
        target_dir=TARGET_SCURVES_DIR,
        output_csv_path=OUTPUT_MATRIX_PATH,
        n_jobs=1,
        prototyping_mode=False,
        resume=True,
        checkpoint_interval_seconds=CHECKPOINT_INTERVAL_SECONDS,
        batch_size=BATCH_SIZE,
        use_tqdm=USE_TQDM,
    )
    del _batch_df
    gc.collect()
    print(f"Batch {_batch_num} complete.")

print("\nAll participant batches complete.")

# Convert the accumulated CSV checkpoint to parquet
_csv_df = pd.read_csv(OUTPUT_MATRIX_PATH)
_csv_df.to_parquet(OUTPUT_MATRIX_PARQUET_PATH, index=False)
del _csv_df

df = pd.read_parquet(OUTPUT_MATRIX_PARQUET_PATH)
print(f"Loaded matrix: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()


Participant batch 1/2: 500 participants
1. Normalising master dataframe schema (6,875,000 rows)...
2. Loading Standardised Target S-Curves...
   Resume mode: loaded 0 processed vase keys.
3. Partitioning dataset into individual vase profiles...
   Successfully isolated 27500 unique mathematical arrays.
4. Engaging parallel extraction across 1 CPU cores...


Extracting batches (single_csv):   0%|          | 0/275 [00:00<?, ?batch/s]

   Checkpoint saved: 100 rows appended to outdir/gpt-5.4-270526/master_feature_matrix_compiled.csv.


KeyboardInterrupt: 

In [ ]:
# Retention check: every vase in the parquet must appear as exactly one row in the output matrix
_input_keys = pd.read_parquet(
    XY_PARQUET_PATH,
    columns=['participant_id', 'generation_key', 'vase'],
).drop_duplicates()
_input_keys = _input_keys.rename(columns={'participant_id': 'user_id', 'generation_key': 'generation'})
_input_vase_ids = (
    _input_keys['user_id'].astype(str) + "::" +
    _input_keys['generation'].astype(str) + "::" +
    _input_keys['vase'].astype(str)
)
_expected_vases = len(_input_vase_ids)

_output_vase_ids = (
    df["user_id"].astype(str) + "::" + df["generation"].astype(str) + "::" + df["vase"].astype(str)
)

_missing = set(_input_vase_ids) - set(_output_vase_ids)
_extra   = set(_output_vase_ids) - set(_input_vase_ids)

print(f"Expected vases (parquet) : {_expected_vases:,}")
print(f"Output rows              : {len(_output_vase_ids):,}")
print(f"Missing from output      : {len(_missing)}")
print(f"Unexpected in output     : {len(_extra)}")

if _missing:
    print("\nSample missing vase_ids:")
    for vid in sorted(_missing)[:10]:
        print(f"  {vid}")

del _input_keys, _input_vase_ids

assert not _missing, f"Retention check FAILED: {len(_missing)} vases missing from output."
print("\nRetention check PASSED.")

In [ ]:
# Diagnostic: how many XY points does each vase profile actually have?
# Load a sample of participants rather than the full parquet to avoid a second
# full copy in memory alongside the extraction results.
_diag_participants = _all_participants[:50].tolist()
_diag = pd.read_parquet(
    XY_PARQUET_PATH,
    filters=[('participant_id', 'in', _diag_participants)],
)
_diag["vase_id"] = (
    _diag["participant_id"].astype(str)
    + "::"
    + _diag["generation_key"].astype(str)
    + "::"
    + _diag["vase"].astype(str)
)
_pts = _diag.groupby('vase_id').size()
print("Points per vase profile:")
print(_pts.value_counts().sort_index().to_string())
print(f"\nUnique vase profiles in sample: {len(_pts)}")
del _diag, _pts

In [ ]:
# Check the number of participants, and the number of completed rounds per participant
_diag_participants = _all_participants[:50].tolist()
_diag = pd.read_parquet(
    XY_PARQUET_PATH,
    columns=['participant_id', 'generation_key'],
    filters=[('participant_id', 'in', _diag_participants)],
)
n_participants = _diag['participant_id'].nunique()
n_choices_per_participant = _diag.groupby('participant_id')['generation_key'].nunique()
print(f"Number of unique participants (sample): {n_participants}")
print("Choices made per participant:")
print(n_choices_per_participant.value_counts().sort_index().to_string())
del _diag

## Step 2: Load Compiled Matrix And Prepare Modelling Inputs
This section loads the compiled matrix and splits columns into:
- administrative columns (identifiers and labels), and
- numeric candidate feature columns for statistical filtering.

`X` is the raw numeric feature matrix. `y` is the binary selection target (`is_selected`).

In [ ]:
admin_cols = ['user_id', 'generation', 'vase', 'is_selected', 'source_csv']
admin_present = [c for c in admin_cols if c in df.columns]

feature_cols = [c for c in df.columns if c not in admin_cols and pd.api.types.is_numeric_dtype(df[c])]

X = df[feature_cols].copy()
y = df['is_selected'].copy()

print(f"   Initial numerical feature space: {len(X.columns)} dimensions.")

In [ ]:
# Create a working copy for filtering/cleaning steps
X_filtered = X.copy()

# Handle edge-case numerics before correlation/clustering
X_filtered.replace([np.inf, -np.inf], np.nan, inplace=True)
X_filtered.fillna(0, inplace=True)

# Drop zero-variance features to avoid undefined correlations
var_mask = X_filtered.var(axis=0, ddof=0) > 0
removed_zero_var = int((~var_mask).sum())
X_filtered = X_filtered.loc[:, var_mask]

print(f"Clean feature matrix ready: {X_filtered.shape[0]} rows x {X_filtered.shape[1]} cols")
print(f"Dropped {removed_zero_var} zero-variance features")

## Step 3: Clean Features Before Correlation
Before correlation analysis, this step ensures numerical stability by:
- replacing infinities,
- filling missing values, and
- removing zero-variance features (which do not carry discriminative information).

The result is `X_filtered`, the cleaned matrix used by all subsequent clustering steps.

In [ ]:
# Calculate non-linear rank correlations
corr_matrix, _ = spearmanr(X_filtered)
corr_df = pd.DataFrame(corr_matrix, columns=X_filtered.columns, index=X_filtered.columns)
corr_df.shape

## Step 4: Build Correlation Distance And Hierarchical Clusters
This stage converts feature similarity into a clustering structure:
- Spearman correlation captures monotonic relationships between features.
- Distance is computed as `1 - |rho|`, so highly correlated pairs have small distance.
- Complete-linkage hierarchical clustering is then used to group redundant features.

The output of this stage is the linkage matrix `Z` and cluster labels `clusters`.

In [ ]:
# Convert Correlation to Distance (1 - |corr|)
distance_matrix = 1 - np.abs(corr_df)
distance_matrix.fillna(1.0, inplace=True) # Max distance for any failed calculations

In [ ]:
# Compute Complete Linkage (Groups clusters based on maximum distance between their points)
condensed_dist = squareform(distance_matrix.values, checks=False)
Z = sch.linkage(condensed_dist, method='complete')

In [ ]:
# Cut the Dendrogram at the specified correlation threshold
corr_threshold = 0.8
dist_threshold = 1.0 - corr_threshold
clusters = sch.fcluster(Z, t=dist_threshold, criterion='distance')

In [ ]:
# Map features to their assigned mathematical clusters
cluster_dict = {}
for i, cluster_id in enumerate(clusters):
    if cluster_id not in cluster_dict:
        cluster_dict[cluster_id] = []
    cluster_dict[cluster_id].append(X_filtered.columns[i])

In [ ]:
# Isolate the Champion Feature from each cluster
selected_features = []
for c_id, cols in cluster_dict.items():
    if len(cols) == 1:
        selected_features.append(cols[0])
    else:
        # The champion is the one most strongly correlated with human selection
        corrs = [np.abs(X_filtered[col].corr(y)) for col in cols]
        corrs = [0 if np.isnan(c) else c for c in corrs]
        best_col = cols[np.argmax(corrs)]
        selected_features.append(best_col)
        
print(f"Dimensionality collapsed to {len(selected_features)} unique signals.")

## Step 5: Select Champion Features Per Cluster
Each collinearity cluster is reduced to one representative feature:
- Single-feature clusters are retained as-is.
- Multi-feature clusters are resolved by choosing the feature with the strongest absolute correlation with `is_selected`.

This yields `selected_features`, the compact signal set used for downstream modelling and interpretation.

In [ ]:
# This dendrogram is purely for visualization and sanity-checking of the clustering results, not for actual feature selection
# It shows the hierarchical relationships between features and where the correlation threshold cut was made, 
# but the actual selection is based on championing within clusters, not on cutting the dendrogram at a specific height
plt.figure(figsize=(20, 10))
sch.dendrogram(Z, labels=X_filtered.columns, leaf_rotation=90, no_labels=True)
plt.axhline(y=dist_threshold, color='r', linestyle='--', label=f'{int(corr_threshold*100)}% Correlation Limit')
plt.title("Spearman Hierarchical Clustering Dendrogram\n(Collinearity Purge)")
plt.ylabel("Distance (1 - |Spearman ρ|)")
plt.legend()
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/dendrogram_purge.png", dpi=300)

In [ ]:
biserial_corrs = []
    
# Calculate absolute Point-Biserial correlation for all purged champion features
for col in selected_features:
    corr = X_filtered[col].corr(y)
    if not np.isnan(corr):
        biserial_corrs.append({'Feature': col, 'Biserial_Correlation': np.abs(corr)})
        
df_corrs = pd.DataFrame(biserial_corrs).sort_values(by='Biserial_Correlation', ascending=False).head(20)

In [ ]:
plt.figure(figsize=(12, 8))
sns.barplot(x='Biserial_Correlation', y='Feature', data=df_corrs, palette='magma')
plt.title("Top 20 Purged Features by Predictive Power\n(Absolute Point-Biserial Correlation)")
plt.xlabel("Absolute Correlation with 'is_selected'")
plt.ylabel("Feature Name")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/plots/top_20_predictive_features.png", dpi=300)

In [ ]:
# Isolate the top 30 most predictive features to prevent the graph from becoming an illegible black mass
top_features = df_corrs['Feature'].tolist()[:30] 
X_top = X_filtered[top_features]
corr_top, _ = spearmanr(X_top)

G = nx.Graph()
for f in top_features:
    G.add_node(f)
    
# Draw edges only if the Spearman correlation exceeds the high-risk threshold
for i in range(len(top_features)):
    for j in range(i+1, len(top_features)):
        if np.abs(corr_top[i, j]) > 0.90: 
            G.add_edge(top_features[i], top_features[j], weight=np.abs(corr_top[i, j]))
            
plt.figure(figsize=(12, 12))
# Spring layout physically groups highly connected nodes
pos = nx.spring_layout(G, k=0.5, seed=42)

nx.draw_networkx_nodes(G, pos, node_size=500, node_color='skyblue', alpha=0.8)

edges = G.edges()
weights = [G[u][v]['weight'] * 2 for u,v in edges]
nx.draw_networkx_edges(G, pos, width=weights, edge_color='gray', alpha=0.5)

nx.draw_networkx_labels(G, pos, font_size=8, font_family='sans-serif')

plt.title("Feature Collinearity Network (Spearman ρ > 0.90)\nNodes clustered by mathematical redundancy")
plt.axis('off')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/plots/collinearity_network.png", dpi=300)

## Step 6: Visualise, Rank, And Export
The final section has three goals:
- Visualise structure: dendrogram and collinearity network plots.
- Rank predictive strength: top champion features by absolute correlation with selection.
- Export deliverables:
  - `master_feature_matrix_purged.csv` (final reduced matrix)
  - saved figures for reporting and team discussion.

Together, these outputs provide both a modelling-ready dataset and an auditable explanation of feature reduction decisions.

In [ ]:
final_cols = admin_present + selected_features
df_final = df[final_cols]
df_final.head()

In [ ]:
df_final.to_csv(f"{OUTPUT_DIR}/legacy_master_feature_matrix_purged.csv", index=False)

# Save the full feature matrix before correlation-based champion selection
df.to_parquet(f"{OUTPUT_DIR}/master_feature_matrix_all_features.parquet", index=False)
print(f"Saved full matrix: {df.shape[0]} rows x {df.shape[1]} columns → {OUTPUT_DIR}/master_feature_matrix_all_features.parquet")
print(f"Saved purged matrix: {df_final.shape[0]} rows x {df_final.shape[1]} columns → {OUTPUT_DIR}/legacy_master_feature_matrix_purged.csv")


## Output Guide
This notebook produces several outputs during extraction, feature pruning, and visualization. Use this guide as a quick reference for what each result means.

### 1) `master_feature_matrix_compiled.csv`
- What it is: the full extracted feature matrix before collinearity pruning.
- Rows: one row per vase profile processed from `xy_csv`.
- Columns: administrative identifiers (`user_id`, `generation`, `vase`, `is_selected`, `source_csv`) plus all computed geometric/relational features.
- Typical use: master input for modeling, statistics, and the pruning pipeline below.

### 2) Console Checkpoint + Process Log (`master_feature_matrix_compiled.csv.process_log.csv`)
- What it is: append-only run log generated during extraction.
- Contains: timestamps, file start/complete events, checkpoint writes, and run summaries.
- Typical use: monitor long runs, verify resume behavior, and audit partial progress.

### 3) In-memory matrices created during pruning
- `X`: all numeric feature columns prior to cleanup.
- `X_filtered`: cleaned feature matrix used for clustering/correlation (NaN/inf handled, zero-variance features removed).
- `corr_df`: Spearman feature-to-feature correlation matrix.
- `distance_matrix`: transformed matrix used for clustering, defined as `1 - |correlation|`.
- `Z`: hierarchical linkage output used to define collinearity clusters.

### 4) `dendrogram_purge.png`
- What it is: dendrogram visualization of hierarchical feature clustering.
- Red dashed line: the cut threshold implied by `corr_threshold` (currently 0.90).
- Typical use: sanity-check redundancy structure and threshold behavior.

### 5) Cluster-selection output
- `selected_features`: one representative ("champion") feature per collinearity cluster.
- Champion rule: for multi-feature clusters, chooses the feature with highest absolute correlation to `is_selected`.
- Printed summary: "Dimensionality collapsed to N unique signals."
- Typical use: reduced feature set for interpretable modeling.

### 6) Top predictive feature ranking (`df_corrs`)
- What it is: top 20 champion features ranked by absolute point-biserial-style correlation with `is_selected`.
- Typical use: quick interpretation of strongest univariate signal contributors.

### 7) `top_20_predictive_features.png`
- What it is: bar chart view of `df_corrs` (top 20 predictive purged features).
- Typical use: presentation-ready summary of strongest selected features.

### 8) `collinearity_network.png`
- What it is: network graph among top predictive features where edges indicate very high Spearman collinearity (`|rho| > 0.90`).
- Edge width: strength of collinearity.
- Typical use: visualize residual redundancy among strongest predictors.

### 9) `master_feature_matrix_purged.csv`
- What it is: final reduced matrix with admin columns + `selected_features`.
- Typical use: compact modeling dataset with reduced multicollinearity.

## Step 8: Build Pairwise XGBoost Dataset (Within-Round Vase Deltas)
This step prepares a pairwise training table for XGBoost from the compiled master matrix.

### What it does
- Parses `generation` into `round_num` and `gen_num`.
- For each user-generation choice event, compares the **two vases in that event**.
- Creates pairwise rows as direct feature deltas (`selected - non_selected`) and the reverse direction.
- Writes a new output CSV ready for pairwise ranking models.

In [ ]:
PAIRWISE_OUTPUT_PATH = f"{OUTPUT_DIR}/pairwise_xgb_within_round_vase_delta_matrix.csv"
PAIRWISE_GROUPS_PATH = f"{OUTPUT_DIR}/pairwise_xgb_within_round_vase_delta_group_sizes.csv"

In [ ]:
df_final.head()

In [ ]:
# 1) Parse round/gen indices from generation labels like round_3_gen_4
pairwise_df = df_final.copy()
if "round" in pairwise_df["generation"].astype(str).str.lower().iloc[0]:
    gen_parts = pairwise_df["generation"].astype(str).str.extract(r"round_(\d+)_gen_(\d+)")
elif "rep" in pairwise_df["generation"].astype(str).str.lower().iloc[0]:
    gen_parts = pairwise_df["generation"].astype(str).str.extract(r"rep_(\d+)_gen_(\d+)")
else:
    gen_parts = pairwise_df["generation"].astype(str).str.extract(r"gen_(\d+)")

if len(gen_parts.columns) < 2:
    gen_parts[1] = gen_parts[0]  # If only one number was extracted, assume it's the generation and set round/rep to 0
    gen_parts[0] = 0
pairwise_df["round_num"] = pd.to_numeric(gen_parts[0], errors="coerce")
pairwise_df["gen_num"] = pd.to_numeric(gen_parts[1], errors="coerce")
pairwise_df = pairwise_df.dropna(subset=["round_num", "gen_num"]).copy()
pairwise_df["round_num"] = pairwise_df["round_num"].astype(int)
pairwise_df["gen_num"] = pairwise_df["gen_num"].astype(int)

pairwise_df.head()

In [ ]:
# 2) Choose base numeric features to delta between vases within the same choice event
exclude_cols = {
    "user_id", "generation", "vase", "is_selected", "source_csv",
    "round_num", "gen_num", "session_id"
}
base_model_cols = [
    c for c in pairwise_df.columns
    if c not in exclude_cols and pd.api.types.is_numeric_dtype(pairwise_df[c])
]

In [ ]:
# 3) Build pairwise rows for same-generation vase choices
# Only keep clean binary choice events with exactly two vases and one selected.
pair_rows = []
group_keys = ["user_id", "generation", "round_num", "gen_num"]
events_seen = 0
events_used = 0
events_skipped = 0

In [ ]:
for key, grp in pairwise_df.groupby(group_keys, sort=False):
    events_seen += 1

    if len(grp) != 2:
        events_skipped += 1
        continue

    selected = grp[grp["is_selected"] == 1]
    not_selected = grp[grp["is_selected"] == 0]

    if len(selected) != 1 or len(not_selected) != 1:
        events_skipped += 1
        continue

    events_used += 1
    s = selected.iloc[0]
    n = not_selected.iloc[0]

    d = s[base_model_cols].to_numpy(dtype=float) - n[base_model_cols].to_numpy(dtype=float)
    query_id = f"{key[0]}::{key[1]}"
    delta_cols = [f"vd_{c}" for c in base_model_cols]

    row_pos = {
        "query_id": query_id,
        "user_id": key[0],
        "generation": key[1],
        "round_num": int(key[2]),
        "gen_num": int(key[3]),
        "left_vase": s["vase"],
        "right_vase": n["vase"],
        "label": 1,
    }
    row_pos.update({col: val for col, val in zip(delta_cols, d)})
    pair_rows.append(row_pos)

    row_neg = {
        "query_id": query_id,
        "user_id": key[0],
        "generation": key[1],
        "round_num": int(key[2]),
        "gen_num": int(key[3]),
        "left_vase": n["vase"],
        "right_vase": s["vase"],
        "label": 0,
    }
    row_neg.update({col: -val for col, val in zip(delta_cols, d)})
    pair_rows.append(row_neg)

pairwise_out = pd.DataFrame(pair_rows)

if pairwise_out.empty:
    raise ValueError(
        "No pairwise rows were created. Verify each generation has exactly two vases and one selected label."
    )

print(f"Pairwise dataset created: {pairwise_out.shape[0]} rows x {pairwise_out.shape[1]} columns")
print(f"Events seen: {events_seen}, used: {events_used}, skipped: {events_skipped}")

In [ ]:
# Drop non-informative delta columns that are always zero (common for metadata constants).
delta_feature_cols = [c for c in pairwise_out.columns if c.startswith("vd_")]
nonzero_feature_cols = [c for c in delta_feature_cols if not np.isclose(pairwise_out[c].abs().sum(), 0.0)]
keep_cols = [
    "query_id", "user_id", "generation", "round_num", "gen_num",
    "left_vase", "right_vase", "label",
] + nonzero_feature_cols
pairwise_out = pairwise_out[keep_cols].copy()

print(f"Final pairwise dataset: {pairwise_out.shape[0]} rows x {pairwise_out.shape[1]} columns")

In [ ]:
pairwise_out.to_csv(PAIRWISE_OUTPUT_PATH, index=False)
group_sizes = pairwise_out.groupby("query_id").size().rename("n_rows").reset_index()
group_sizes.to_csv(PAIRWISE_GROUPS_PATH, index=False)

print(f"Pairwise file written: {PAIRWISE_OUTPUT_PATH}")
print(f"Group-size file written: {PAIRWISE_GROUPS_PATH}")
print(f"Events seen: {events_seen} | used: {events_used} | skipped: {events_skipped}")
print(f"Pairwise shape: {pairwise_out.shape}")
print(f"Unique queries: {group_sizes.shape[0]}")
print(f"Delta features kept: {len(nonzero_feature_cols)}")

# Mirrored pairwise has 2 rows per decision event; decisions = total vases / 2
_expected_decisions = _expected_vases // 2
_expected_mirrored_rows = _expected_decisions * 2
assert len(pairwise_out) == _expected_mirrored_rows, (
    f"Mirrored pairwise FAILED: expected {_expected_mirrored_rows:,} rows "
    f"({_expected_decisions:,} decisions × 2), got {len(pairwise_out):,}."
)
print(f"\nMirrored pairwise check PASSED: {_expected_decisions:,} decisions → {_expected_mirrored_rows:,} rows.")

In [ ]:
# Diagnostic: show skipped generation-events and reasons
skip_rows = []
diag_keys = ["user_id", "generation", "round_num", "gen_num"]

In [ ]:
pairwise_df.head()

In [ ]:
for key, grp in pairwise_df.groupby(diag_keys, sort=False):
    n_rows = len(grp)
    n_selected = int((grp["is_selected"] == 1).sum())
    n_not_selected = int((grp["is_selected"] == 0).sum())

    reason = None
    if n_rows != 2:
        reason = f"expected_2_vases_found_{n_rows}"
    elif n_selected != 1 or n_not_selected != 1:
        reason = f"invalid_labels_selected_{n_selected}_not_selected_{n_not_selected}"

    if reason is not None:
        skip_rows.append({
            "user_id": key[0],
            "generation": key[1],
            "round_num": int(key[2]),
            "gen_num": int(key[3]),
            "n_vases": n_rows,
            "n_selected": n_selected,
            "n_not_selected": n_not_selected,
            "reason": reason,
            "vases": ", ".join(sorted(grp["vase"].astype(str).unique())),
        })
        
print(f"Total skipped events: {len(skip_rows)}")

In [ ]:
if skip_rows:
    skipped_events_df = pd.DataFrame(skip_rows).sort_values(["user_id", "round_num", "gen_num"])
else:    skipped_events_df = pd.DataFrame(columns=[
        "user_id", "generation", "round_num", "gen_num",
        "n_vases", "n_selected", "n_not_selected",
        "reason", "vases"
    ])

skipped_events_df.head()

In [ ]:
if skipped_events_df.empty:
    print("No skipped events.")
else:
    display(skipped_events_df)
    display(skipped_events_df["reason"].value_counts().rename_axis("reason").to_frame("count"))

In [ ]:
# Preview the generated pairwise dataset
pairwise_out.head()

## Step 9: Optimal XGBoost Implementations (In Addition To Current Setup)
This section keeps the existing mirrored pairwise dataset and adds two optimized exports:

1. **Compact Pairwise Classification (one row per choice event)**
   - Uses deterministic vase ordering (`left_vase`, `right_vase`) within each event.
   - Label is `1` if the ordered left vase was selected, else `0`.
   - Feature deltas are `left - right` with prefix `vd_`.

2. **Native XGBoost Ranking Format (`rank:pairwise`)**
   - One row per item (vase), not a delta row.
   - `query_id` groups two rows per event.
   - `label` is relevance (`1` selected, `0` not selected).

This gives you both your current mirrored structure and an efficient/standard ranking-ready alternative.

### Step 9 Output Guide (Delta vs Raw-Value XGBoost)
Use this section as a quick decision guide when running both model families.

#### A) Mirrored Delta Pairwise Dataset (current setup)
- File: `outdir/pairwise_xgb_within_round_vase_delta_matrix.csv`
- Structure: two rows per choice event (selected-minus-rejected and reversed).
- Target column: `label` (1/0).
- Best used for: pairwise **classification** workflows (`binary:logistic`) where you want strict comparison symmetry.

#### B) Compact Delta Pairwise Dataset (optimised delta setup)
- File: `pairwise_xgb_compact_one_row_per_event.csv`
- Structure: one deterministic row per choice event using `left_vase - right_vase`.
- Target column: `left_selected` (1 if left vase was selected, 0 otherwise).
- Best used for: efficient pairwise **classification** with easier data auditing and fewer duplicated rows.

#### C) Raw-Value Rank-Item Dataset (optimised ranking setup)
- File: `xgb_rank_items_within_round.csv`
- Structure: one row per vase item, grouped by `query_id` (typically two rows per event).
- Target column: `relevance` (1 selected, 0 not selected).
- Best used for: native XGBoost **ranking** objective (`rank:pairwise`) and SHAP interpretation on original vase features.

#### Group Files
- `outdir/pairwise_xgb_within_round_vase_delta_group_sizes.csv`
- `pairwise_xgb_compact_group_sizes.csv`
- `xgb_rank_items_within_round_group_sizes.csv`
- These list rows per `query_id` and are useful for validation, diagnostics, and any training pipeline that requires explicit group sizes.

#### Practical Recommendation
- Run both families, as planned:
  - Delta model for comparative decision mechanics.
  - Raw-value rank model for feature-native interpretation and ranking behaviour.
- Compare performance and explanation stability across both to identify robust beauty preference signals.

In [ ]:
# Step 9A) Build a compact one-row-per-event pairwise classification dataset
PAIRWISE_COMPACT_OUTPUT_PATH = f"{OUTPUT_DIR}/legacy/pairwise_xgb_compact_one_row_per_event.csv"
PAIRWISE_COMPACT_GROUPS_PATH = f"{OUTPUT_DIR}/legacy/pairwise_xgb_compact_group_sizes.csv"

compact_rows = []
compact_skipped = 0

for key, grp in pairwise_df.groupby(group_keys, sort=False):
    if len(grp) != 2:
        compact_skipped += 1
        continue

    selected = grp[grp["is_selected"] == 1]
    not_selected = grp[grp["is_selected"] == 0]
    if len(selected) != 1 or len(not_selected) != 1:
        compact_skipped += 1
        continue

    # Deterministic orientation so we only need one row per event
    ordered = grp.sort_values("vase").reset_index(drop=True)
    left = ordered.iloc[0]
    right = ordered.iloc[1]
    left_selected = int(left["is_selected"] == 1)

    delta = left[base_model_cols].to_numpy(dtype=float) - right[base_model_cols].to_numpy(dtype=float)
    delta_cols = [f"vd_{c}" for c in base_model_cols]

    row = {
        "query_id": f"{key[0]}::{key[1]}",
        "user_id": key[0],
        "generation": key[1],
        "round_num": int(key[2]),
        "gen_num": int(key[3]),
        "left_vase": left["vase"],
        "right_vase": right["vase"],
        "left_selected": left_selected,
    }
    row.update({c: v for c, v in zip(delta_cols, delta)})
    compact_rows.append(row)

pairwise_compact = pd.DataFrame(compact_rows)
if pairwise_compact.empty:
    raise ValueError("Compact pairwise dataset is empty. Check event structure/labels.")

# Drop all-zero delta features
compact_delta_cols = [c for c in pairwise_compact.columns if c.startswith("vd_")]
compact_nonzero = [c for c in compact_delta_cols if not np.isclose(pairwise_compact[c].abs().sum(), 0.0)]
pairwise_compact = pairwise_compact[[
    "query_id", "user_id", "generation", "round_num", "gen_num",
    "left_vase", "right_vase", "left_selected",
] + compact_nonzero].copy()

pairwise_compact.to_csv(PAIRWISE_COMPACT_OUTPUT_PATH, index=False)
pairwise_compact_groups = pairwise_compact.groupby("query_id").size().rename("n_rows").reset_index()
pairwise_compact_groups.to_csv(PAIRWISE_COMPACT_GROUPS_PATH, index=False)

print(f"Compact pairwise file: {PAIRWISE_COMPACT_OUTPUT_PATH}")
print(f"Compact group-size file: {PAIRWISE_COMPACT_GROUPS_PATH}")
print(f"Compact shape: {pairwise_compact.shape}")
print(f"Compact skipped events: {compact_skipped}")
print(f"Compact delta features kept: {len(compact_nonzero)}")

# Compact has exactly 1 row per decision event; decisions = total vases / 2
assert len(pairwise_compact) == _expected_decisions, (
    f"Compact pairwise FAILED: expected {_expected_decisions:,} rows (one per decision), "
    f"got {len(pairwise_compact):,}."
)
print(f"\nCompact pairwise check PASSED: {_expected_decisions:,} decisions → {len(pairwise_compact):,} rows.")

In [ ]:
# Step 9A diagnostics: which compact delta features were dropped and why
compact_dropped_features = sorted(set(compact_delta_cols) - set(compact_nonzero))

compact_drop_report = pd.DataFrame({
    "feature": compact_dropped_features,
    "reason": "Absolute sum across all events is 0 (no variation in compact left-right deltas).",
})

print(f"Compact features dropped: {len(compact_dropped_features)}")
if compact_drop_report.empty:
    print("No compact delta features were dropped.")
else:
    display(compact_drop_report)

In [ ]:
# Step 9B) Build native item-level ranking dataset for XGBoost rank:pairwise
RANK_ITEMS_OUTPUT_PATH = f"{OUTPUT_DIR}/legacy/xgb_rank_items_within_round.csv"
RANK_GROUPS_OUTPUT_PATH = f"{OUTPUT_DIR}/legacy/xgb_rank_items_within_round_group_sizes.csv"

rank_rows = []
rank_skipped = 0

for key, grp in pairwise_df.groupby(group_keys, sort=False):
    if len(grp) != 2:
        rank_skipped += 1
        continue

    selected = grp[grp["is_selected"] == 1]
    not_selected = grp[grp["is_selected"] == 0]
    if len(selected) != 1 or len(not_selected) != 1:
        rank_skipped += 1
        continue

    qid = f"{key[0]}::{key[1]}"
    for _, r in grp.iterrows():
        row = {
            "query_id": qid,
            "user_id": key[0],
            "generation": key[1],
            "round_num": int(key[2]),
            "gen_num": int(key[3]),
            "vase": r["vase"],
            "relevance": int(r["is_selected"]),
        }
        for c in base_model_cols:
            row[c] = r[c]
        rank_rows.append(row)

rank_items = pd.DataFrame(rank_rows)
if rank_items.empty:
    raise ValueError("Ranking item-level dataset is empty. Check event structure/labels.")

# Drop all-zero numeric item features
rank_numeric_features = [c for c in base_model_cols if c in rank_items.columns]
rank_nonzero = [c for c in rank_numeric_features if not np.isclose(rank_items[c].abs().sum(), 0.0)]
rank_items = rank_items[[
    "query_id", "user_id", "generation", "round_num", "gen_num", "vase", "relevance",
] + rank_nonzero].copy()

rank_items.to_csv(RANK_ITEMS_OUTPUT_PATH, index=False)
rank_groups = rank_items.groupby("query_id").size().rename("n_rows").reset_index()
rank_groups.to_csv(RANK_GROUPS_OUTPUT_PATH, index=False)

print(f"Rank-item file: {RANK_ITEMS_OUTPUT_PATH}")
print(f"Rank-item group-size file: {RANK_GROUPS_OUTPUT_PATH}")
print(f"Rank-item shape: {rank_items.shape}")
print(f"Rank-item skipped events: {rank_skipped}")
print(f"Rank-item features kept: {len(rank_nonzero)}")

# Rank-item has 2 rows per decision event (one per vase); decisions = total vases / 2
_expected_rank_rows = _expected_vases  # 2 items × (vases/2) decisions = vases
assert len(rank_items) == _expected_rank_rows, (
    f"Rank-item FAILED: expected {_expected_rank_rows:,} rows "
    f"({_expected_decisions:,} decisions × 2 items), got {len(rank_items):,}."
)
print(f"\nRank-item check PASSED: {_expected_decisions:,} decisions → {len(rank_items):,} rows.")

In [ ]:
# Step 9B diagnostics: which raw-value rank features were dropped and why
rank_dropped_features = sorted(set(rank_numeric_features) - set(rank_nonzero))

rank_drop_report = pd.DataFrame({
    "feature": rank_dropped_features,
    "reason": "Absolute sum across all ranking rows is 0 (no variation / constant zero feature).",
})

print(f"Rank-item features dropped: {len(rank_dropped_features)}")
if rank_drop_report.empty:
    print("No rank-item features were dropped.")
else:
    display(rank_drop_report)